In [1]:
!pip install -q --upgrade transformers datasets peft bitsandbytes trl
!pip install -q accelerate
from accelerate.utils import write_basic_config
write_basic_config()

PosixPath('/root/.cache/huggingface/accelerate/default_config.yaml')

In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
from itertools import product
from tqdm import tqdm

import torch
from torch.utils.data import DataLoader

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from datasets import load_dataset
from huggingface_hub import login


In [3]:
seed = 7
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [4]:
token = ""
login(token)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [5]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

In [6]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
tokenizer.padding_side = 'left' 
tokenizer.truncation_side='right'

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [7]:
DEFAULT_PAD_TOKEN = "[PAD]"
DEFAULT_EOS_TOKEN = "</s>"
DEFAULT_BOS_TOKEN = "<s>"

special_tokens_dict = {}
if tokenizer.pad_token is None:
    special_tokens_dict['pad_token'] = DEFAULT_PAD_TOKEN
if tokenizer.eos_token is None:
    special_tokens_dict['eos_token'] = DEFAULT_EOS_TOKEN
if tokenizer.bos_token is None:
    special_tokens_dict['bos_token'] = DEFAULT_BOS_TOKEN

if special_tokens_dict:
    tokenizer.add_special_tokens(special_tokens_dict)

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=token
)

model.resize_token_embeddings(len(tokenizer))
model.eval()  

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128257, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-0

In [9]:
dataset = load_dataset("lighteval/MATH", trust_remote_code=True)
test_dataset = dataset['test']

MATH.py:   0%|          | 0.00/4.10k [00:00<?, ?B/s]

data/algebra_train.jsonl:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

(…)ata/counting_and_probability_train.jsonl:   0%|          | 0.00/707k [00:00<?, ?B/s]

data/geometry_train.jsonl:   0%|          | 0.00/1.15M [00:00<?, ?B/s]

data/intermediate_algebra_train.jsonl:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

...

data/number_theory_test.jsonl:   0%|          | 0.00/376k [00:00<?, ?B/s]

data/prealgebra_test.jsonl:   0%|          | 0.00/553k [00:00<?, ?B/s]

data/precalculus_test.jsonl:   0%|          | 0.00/614k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [10]:
problems = []
ground_truths = []
levels = []
problem_types = []
input_texts = []

for sample in test_dataset:
    problem = sample['problem']
    ground_truth = sample['solution']
    level = sample.get('level', 'unknown')
    problem_type = sample.get('type', 'unknown')

    input_text = (
        "<|begin_of_text|><|start_header_id|>system <|end_header_id|>"
        "You are an expert math assistant<|eot_id|><|start_header_id|>user <|end_header_id|>"
        f"Solve the following math problem: {problem}\n"
        "Show all intermediate steps and please mandatorily include the final answer in LaTeX format in a box like \\boxed{{}}."
        "<|eot_id|><|start_header_id|> assistant <|end_header_id|>"
    )

    input_texts.append(input_text)
    problems.append(problem)
    ground_truths.append(ground_truth)
    levels.append(level)
    problem_types.append(problem_type)

In [11]:
token_lengths = []
for solution in tqdm(ground_truths, desc="Calculating solution lengths"):
    tokens = tokenizer.encode(solution, add_special_tokens=False)
    token_lengths.append(len(tokens))

min_length = np.min(token_lengths)
max_length = np.max(token_lengths)
average_length = np.mean(token_lengths)
median_length = np.median(token_lengths)
percentile_90 = np.percentile(token_lengths, 90)
percentile_95 = np.percentile(token_lengths, 95)
percentile_99 = np.percentile(token_lengths, 99)

print(f"Minimum length: {min_length}")
print(f"Maximum length: {max_length}")
print(f"Average length: {average_length}")
print(f"Median length: {median_length}")
print(f"90th percentile length: {percentile_90}")
print(f"95th percentile length: {percentile_95}")
print(f"99th percentile length: {percentile_99}")


Calculating solution lengths:   0%|          | 0/5000 [00:00<?, ?it/s]


Calculating solution lengths:   6%|▌         | 285/5000 [00:00<00:01, 2846.62it/s]


Calculating solution lengths:  13%|█▎        | 629/5000 [00:00<00:01, 3188.45it/s]


Calculating solution lengths:  19%|█▉        | 966/5000 [00:00<00:01, 3268.97it/s]


Calculating solution lengths:  26%|██▌       | 1305/5000 [00:00<00:01, 3314.45it/s]

...


Calculating solution lengths:  85%|████████▍ | 4245/5000 [00:01<00:00, 3060.13it/s]


Calculating solution lengths:  91%|█████████ | 4558/5000 [00:01<00:00, 2966.78it/s]


Calculating solution lengths:  97%|█████████▋| 4860/5000 [00:01<00:00, 2533.11it/s]


Calculating solution lengths: 100%|██████████| 5000/5000 [00:01<00:00, 2550.50it/s]

Minimum length: 15
Maximum length: 1929
Average length: 206.0978
Median length: 146.0
90th percentile length: 430.0
95th percentile length: 581.0500000000002
99th percentile length: 926.1100000000024


In [12]:
def collate_fn(batch):
    # Tokenize the batch of inputs with padding
    model_inputs = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=512,  
        return_tensors="pt"
    )

    model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}
    return model_inputs

In [13]:
batch_size = 128  # Adjust batch size based on your GPU memory
test_dataloader = DataLoader(input_texts, batch_size=batch_size, collate_fn=collate_fn)

In [14]:
parameter_grid = {
    'do_sample': [True, False],
    'temperature': [0.3, 1.7],
    'top_p': [0.5, 0.9],
    'top_k': [10, 100]
}

In [15]:
sampling_params = []
for temp, top_p, top_k in product(
    parameter_grid['temperature'],
    parameter_grid['top_p'],
    parameter_grid['top_k']
):
    sampling_params.append({
        'temperature': temp,
        'top_p': top_p,
        'top_k': top_k
    })

In [16]:
sampling_params

[{'temperature': 0.3, 'top_p': 0.5, 'top_k': 10},
 {'temperature': 0.3, 'top_p': 0.5, 'top_k': 100},
 {'temperature': 0.3, 'top_p': 0.9, 'top_k': 10},
 {'temperature': 0.3, 'top_p': 0.9, 'top_k': 100},
 {'temperature': 1.7, 'top_p': 0.5, 'top_k': 10},
 {'temperature': 1.7, 'top_p': 0.5, 'top_k': 100},
 {'temperature': 1.7, 'top_p': 0.9, 'top_k': 10},
 {'temperature': 1.7, 'top_p': 0.9, 'top_k': 100}]

In [17]:
def run_experiment(do_sample, temperature=None, top_p=None, top_k=None):
    results_list = []

    # Loop over the test DataLoader
    for batch_idx, model_inputs in enumerate(tqdm(test_dataloader, desc="Evaluating")):
        batch_start_idx = batch_idx * batch_size
        current_batch_size = model_inputs['input_ids'].size(0)

        # Prepare generation arguments
        gen_kwargs = {
            'input_ids': model_inputs['input_ids'],
            'attention_mask': model_inputs['attention_mask'],
            'max_new_tokens': 1024,
            'do_sample': do_sample,
            'repetition_penalty': 1.1,
            'eos_token_id': tokenizer.eos_token_id,
            'pad_token_id': tokenizer.pad_token_id,
            'num_beams': 1  # Greedy decoding when do_sample=False
        }

        if do_sample:
            gen_kwargs.update({
                'temperature': temperature,
                'top_p': top_p,
                'top_k': top_k if top_k > 0 else None  # Disable top_k if set to 0
            })

        # Generate predictions
        try:
            with torch.no_grad():
                output_ids = model.generate(**gen_kwargs)
            # Decode the outputs
            for i in range(current_batch_size):
                predicted_text = tokenizer.decode(output_ids[i], skip_special_tokens=True)
                # Store the results
                results_list.append({
                    "problem": problems[batch_start_idx + i],
                    "level": levels[batch_start_idx + i],
                    "type": problem_types[batch_start_idx + i],
                    "ground_truth": ground_truths[batch_start_idx + i],
                    "predicted_solution": predicted_text,
                    "do_sample": do_sample,
                    "temperature": temperature if do_sample else None,
                    "top_p": top_p if do_sample else None,
                    "top_k": top_k if do_sample else None
                })
        except Exception as e:
            print(f"Error during generation: {e}")
            # Handle errors by recording empty predictions
            for i in range(current_batch_size):
                results_list.append({
                    "problem": problems[batch_start_idx + i],
                    "level": levels[batch_start_idx + i],
                    "type": problem_types[batch_start_idx + i],
                    "ground_truth": ground_truths[batch_start_idx + i],
                    "predicted_solution": "",
                    "do_sample": do_sample,
                    "temperature": temperature if do_sample else None,
                    "top_p": top_p if do_sample else None,
                    "top_k": top_k if do_sample else None
                })
            continue
        print(batch_idx)

    return results_list

In [18]:
experiment_counter = 1

# Run experiments with do_sample=False (Deterministic Decoding)
print("Running experiments with do_sample=False (Deterministic Decoding)")
results = run_experiment(do_sample=False)
# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(f"experiment_{experiment_counter}_results.csv", index=False)
print(f"Saved results for experiment {experiment_counter}")
experiment_counter += 1

# Run experiments with do_sample=True (Stochastic Sampling)
print("Running experiments with do_sample=True (Stochastic Sampling)")
for params in sampling_params:
    temperature = params['temperature']
    top_p = params['top_p']
    top_k = params['top_k']
    print(f"\nExperiment {experiment_counter}: temperature={temperature}, top_p={top_p}, top_k={top_k}")
    results = run_experiment(
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    # Save results to a CSV file
    results_df = pd.DataFrame(results)
    results_df.to_csv(f"experiment_{experiment_counter}_results.csv", index=False)
    print(f"Saved results for experiment {experiment_counter}")
    experiment_counter += 1

Running experiments with do_sample=False (Deterministic Decoding)



Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



Evaluating:   2%|▎         | 1/40 [06:26<4:11:18, 386.62s/it]

0


...


Evaluating:  65%|██████▌   | 26/40 [3:01:59<1:27:40, 375.76s/it]

25



Evaluating:  68%|██████▊   | 27/40 [3:07:40<1:19:12, 365.59s/it]

26



Evaluating:  70%|███████   | 28/40 [3:13:20<1:11:34, 357.88s/it]